# 0.22 — Quantum theme: entity-emergence pipeline (0.20 algorithm, re-windowed)

The 0.19/0.20 pipeline applied to **quantum computing**. Benchmark = **WQTM** (first pure
quantum ETF, **2025-10-09**). Baseline = **2019–2020** (so the 2021 pure-plays read as novel);
discovery = **2021–2023**.

**Two honest constraints found upfront:**
- Stage 0 uses the **manual blocklist**, not NER: the NER vocab was genAI-window-specific, and
  `quantum` is a dictionary word (the non-dictionary trick can't keep it).
- Only **`ionq`** is genuinely novel vs 2019–20 (`rigetti`/`d-wave`/`quantum`/`quantum computing`
  already existed). So this leans on IonQ's 2021 SPAC emergence as the anchor, with the (old)
  quantum ecosystem as partners — a thinner, sparser signal than genAI.

In [1]:
import os, re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import networkx as nx, numpy as np, pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"
ENV_PATH = _ROOT / ".env"
if ENV_PATH.exists():
    for _l in ENV_PATH.read_text().splitlines():
        _l=_l.strip()
        if "=" in _l and not _l.startswith("#"):
            _k,_,_v=_l.partition("="); os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

BASELINE_END = pd.Timestamp("2020-12-31")
DISCOVERY_START = pd.Timestamp("2021-01-01")
DISCOVERY_END = pd.Timestamp("2023-12-31")
QTUM = pd.Timestamp("2018-09-04")          # first quantum ETF (broad; pre-dates the news)
WQTM = pd.Timestamp("2025-10-09")          # first PURE quantum ETF -> benchmark
INCEPTION = WQTM
FREQ="W-MON"; REP_HL=8
MIN_MENTIONS=3; DEGREE_MIN=8; PERSIST_WEEKS=2; CLUSTER_K=15; CLUSTER_MAX=0.60; LLM_MAX_GROUPS=25

TERM_STOP = set(ENGLISH_STOP_WORDS) | {"says","said","new","year","week","report","shares","stock",
    "jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"}
EVENT_STOP = {"collapse","rout","bankruptcy","bankrupt","fraud","lawsuit","sue","sues","sued","probe",
    "hearing","trial","court","arrest","arrested","resign","resigns","ban","bans","banned","outage","recall",
    "default","slump","slumps","plunge","plunges","crash","derailment","strike","quake","earthquake","protest",
    "protests","unrest","attack","war","sanctions","fine","fined","scandal","layoffs","layoff","pivot","halt",
    "halts","delay","delays","death","dies","killed","guilty","charges","charged","indicted","crisis","takeover",
    "merger","deal","acquires","acquire","buys","stake","ipo","listing","bond","bonds","notes","debt","offering",
    "case","settlement","shortage","shutdown","tumbles","soars","jumps","rises","falls","drops","gains","cuts","raises"}
def is_entity(t): return not any(tok in EVENT_STOP for tok in t.split())   # Stage 0 = manual blocklist
# quantum validation lexicon (annotation only)
HINT = re.compile(r"\bionq\b|rigetti|\bd-wave\b|\bdwave\b|quantinuum|\bqubit|quantum comput", re.I)

def normalize_terms(v): return v.tolist() if isinstance(v, np.ndarray) else (list(v) if isinstance(v,(list,tuple)) else [])
def filter_terms(ts): return [t for t in ts if len(t)>=3 and not any(tok in TERM_STOP for tok in t.split())]
def week_ts(w): return pd.Period(w, freq=FREQ).start_time
def clustcoef(P, adj):
    if len(P)<2: return 0.0
    tot=pairs=0
    for i in range(len(P)):
        for j in range(i+1,len(P)):
            tot+=1
            if P[j] in adj[P[i]]: pairs+=1
    return pairs/tot if tot else 0.0
print("setup ok · baseline <=", BASELINE_END.date(), "· discovery", DISCOVERY_START.date(),"->",DISCOVERY_END.date())

setup ok · baseline <= 2020-12-31 · discovery 2021-01-01 -> 2023-12-31


In [2]:
def detect(news, baseline_end, ds, de):
    df = news.copy()
    df["terms_f"] = df["terms"].map(normalize_terms).map(filter_terms)
    df["week"] = df["date"].dt.to_period(FREQ).astype(str)
    baseline_terms=set()
    for tl in df.loc[df.date<=baseline_end,"terms_f"]: baseline_terms.update(tl)
    disc=[w for w in sorted(df["week"].unique(), key=week_ts) if ds<=week_ts(w)<=de]
    rows=[]
    for week in disc:
        grp=df.loc[df.week==week]
        twc,adj=Counter(),defaultdict(Counter)
        for tl in grp["terms_f"]:
            s=set(tl)
            for t in s: twc[t]+=1
            for a,b in combinations(sorted(s),2): adj[a][b]+=1; adj[b][a]+=1
        for t,cnt in twc.items():
            if t in baseline_terms or cnt<MIN_MENTIONS: continue
            if not is_entity(t): continue
            epart=[p for p,_ in adj[t].most_common() if is_entity(p)]
            P=epart[:CLUSTER_K]
            reps=[hl for hl,tl in zip(grp["Headline"],grp["terms_f"]) if t in set(tl)][:REP_HL]
            rows.append({"week":week,"anchor":t,"mentions":int(cnt),"anchor_degree":len(epart),
                         "clustering":round(clustcoef(P,adj),3),"subgraph":[t]+P[:10],"rep_headlines":reps})
    return pd.DataFrame(rows)

In [3]:
# load 2019-2020 baseline + 2021-2023 discovery, concat
base = pd.read_parquet(OUTPUT_DIR/"baseline_2019_2020_terms.parquet")
disc = pd.read_parquet(OUTPUT_DIR/"genai_graph_terms.parquet")[["Headline","date","terms"]]
base["date"]=pd.to_datetime(base["date"]).dt.normalize()
disc["date"]=pd.to_datetime(disc["date"]).dt.normalize()
news = pd.concat([base, disc], ignore_index=True)
news = news[news.date<=DISCOVERY_END]
print(f"corpus: {len(news):,} headlines ({news.date.min().date()} -> {news.date.max().date()})")

R = detect(news, BASELINE_END, DISCOVERY_START, DISCOVERY_END)
R["q"] = R["anchor"].str.contains(HINT, na=False)
print(f"{len(R):,} (novel entity x week) rows · {R.anchor.nunique():,} distinct novel entities")
print("\nquantum-lexicon novel anchors (mentions/degree/clustering):")
print(R.loc[R.q,["week","anchor","mentions","anchor_degree","clustering"]].sort_values(["week","anchor"]).head(25).to_string(index=False))

corpus: 6,974,403 headlines (2019-01-01 -> 2023-12-30)
79,087 (novel entity x week) rows · 47,042 distinct novel entities

quantum-lexicon novel anchors (mentions/degree/clustering):
                 week         anchor  mentions  anchor_degree  clustering
2021-02-23/2021-03-01           ionq         4              5       1.000
2021-03-02/2021-03-08           ionq         4             12       0.439
2021-09-21/2021-09-27           ionq         3             14       0.538
2021-09-28/2021-10-04           ionq         5             13       0.692
2021-11-16/2021-11-22           ionq         4             19       0.314
2022-05-03/2022-05-09           ionq         3             12       0.515
2022-08-09/2022-08-15           ionq         7             12       0.394
2022-08-16/2022-08-22           ionq         3             11       0.436
2022-11-08/2022-11-14 d-wave quantum         6             20       0.524
2022-11-08/2022-11-14           ionq         7             24       0.438
202

In [4]:
R["caught"]=(R.mentions>=MIN_MENTIONS)&(R.anchor_degree>=DEGREE_MIN)
def promote(R):
    out=[]
    for a,sub in R[R.caught].groupby("anchor"):
        sub=sub.sort_values("week", key=lambda s:s.map(week_ts))
        wk,deg,clu=list(sub.week),list(sub.anchor_degree),list(sub.clustering)
        p=None
        for i in range(len(wk)):
            if (i+1)>=PERSIST_WEEKS and deg[i]>=max(deg[:i] or [0]) and float(np.median(clu[:i+1]))<=CLUSTER_MAX:
                p=wk[i]; break
        out.append({"anchor":a,"first_caught":wk[0],"n_weeks":len(wk),"deg_max":max(deg),
                    "med_clustering":round(float(np.median(clu)),3),"promoted_week":p,"q":bool(HINT.search(a))})
    return pd.DataFrame(out)
P=promote(R); promoted=P[P.promoted_week.notna()].copy()
print(f"caught {R[R.caught].anchor.nunique():,} · promoted {len(promoted)}")
print("\nquantum-lexicon lifecycle:")
print(P[P.q].sort_values("first_caught")[["anchor","first_caught","n_weeks","deg_max","med_clustering","promoted_week"]].head(15).to_string(index=False))

caught 37,244 · promoted 1312

quantum-lexicon lifecycle:
        anchor          first_caught  n_weeks  deg_max  med_clustering         promoted_week
          ionq 2021-03-02/2021-03-08       16       34           0.423 2021-09-21/2021-09-27
d-wave quantum 2022-11-08/2022-11-14        2       20           0.576                  None


In [5]:
from typing import Literal
from pydantic import BaseModel
pset=set(promoted.anchor)
G=nx.Graph(); G.add_nodes_from(pset)
for _,r in R[R.anchor.isin(pset)&R.caught].iterrows():
    for p in r.subgraph:
        if p in pset and p!=r.anchor: G.add_edge(r.anchor,p)
groups=[sorted(c) for c in nx.connected_components(G)]
def members_subgraph(anchors):
    s=set()
    for sg in R[R.anchor.isin(anchors)&R.caught].subgraph: s.update(sg)
    return sorted(s)
def diverse_evidence(anchors):
    seen,out=set(),[]
    for h in R[R.anchor.isin(anchors)&R.caught].sort_values("week",key=lambda s:s.map(week_ts)).rep_headlines:
        for hl in h:
            if hl.lower() not in seen: seen.add(hl.lower()); out.append(hl)
    return out[::max(1,len(out)//10)][:10] if len(out)>10 else out
gdf=pd.DataFrame({"anchors":groups})
gdf["reach"]=gdf.anchors.map(lambda a:int(P.set_index("anchor").loc[a,"deg_max"].max()))
gdf["q"]=gdf.anchors.map(lambda a:any(HINT.search(x) for x in a))
gdf["subgraph"]=gdf.anchors.map(members_subgraph)
gdf=gdf.sort_values("reach",ascending=False).reset_index(drop=True)
REJECT_SYS=("You are a conservative FILTER that REMOVES clusters of news headlines that are NOT emerging themes. "
 "You never decide what IS a theme; only flag clusters that CLEARLY are: 1. single_entity_event, "
 "2. macro_market_aggregate, 3. boilerplate_wire. If unclear, KEEP. KEEP anything describing a SPECIFIC "
 "technological/industrial/product development across multiple actors.")
class Reject(BaseModel):
    verdict: Literal["keep","reject"]; category: Literal["single_entity_event","macro_market_aggregate","boilerplate_wire","none"]; reason: str
_client=None
def judge(sg,hl):
    global _client
    from openai import OpenAI
    if _client is None: _client=OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ.get("OPENAI_BASE") or None)
    user="Cluster entities: "+", ".join(sg[:14])+"\nHeadlines:\n"+"\n".join(f"- {h}" for h in hl)+"\n\nClassify this cluster."
    r=_client.beta.chat.completions.parse(model=os.environ.get("OPENAI_DEFAULT_MODEL","gpt-4o-mini"),temperature=0,
        response_format=Reject, messages=[{"role":"system","content":REJECT_SYS},{"role":"user","content":user}])
    p=r.choices[0].message.parsed; return (p.verdict=="keep"),p.category
gdf["llm_keep"],gdf["llm_cat"]=None,None
for i,g in gdf.head(LLM_MAX_GROUPS).iterrows():
    k,c=judge(g.subgraph, diverse_evidence(g.anchors)); gdf.at[i,"llm_keep"],gdf.at[i,"llm_cat"]=k,c
print(f"assembled {len(groups)} themes · LLM kept {int((gdf.llm_keep==True).sum())}")

assembled 460 themes · LLM kept 2


In [8]:
gdf

,anchors,reach,q,subgraph,llm_keep,llm_cat
0,"[abb sees, abdullah al-othaim, abortion-pill, ...",3901,True,"[a-1, a-2, a-3, a-mark, a-mark precious, a-sha...",True,none
1,"[raimondo, srettha]",120,False,"[abc, act, action, addresses, affairs, afterno...",True,none
2,[kherson],86,False,"[amid, amid kherson, authorities, captured, ci...",False,none
3,"[economics daily, higher-for-longer]",85,False,"[accidents, adds, africa, aims, alarm, america...",False,none
4,[russian invasion],79,False,"[american, amid, average, average share, bofa,...",False,none
...,...,...,...,...,...,...
455,[inselspital-stiftung],9,False,"[chf, chf100m, chf250m, chf250m green, chf260m...",None,None
456,[wonderplanet],9,False,"[atsushi, atsushi ishikawa, brain, cool, cool ...",None,None
457,[humbl],8,False,"[acquisition, agora, agreement, completion, di...",None,None
458,[bwgi],8,False,"[acquisition, agrees, agricole, bwgi, bwgi acq...",None,None


In [6]:
qc=R[R.q&R.caught]
qfirst=qc.week.min() if len(qc) else None
gp=P[P.q&P.promoted_week.notna()]; qprom=gp.promoted_week.min() if len(gp) else None
print("="*68)
print("QUANTUM benchmark (no keywords at detection):")
print(f"  first CAUGHT (quantum entity) : {qfirst}")
print(f"  PROMOTED                      : {qprom}")
print(f"  QTUM (broad ETF) 2018-09-04 · WQTM (pure ETF) {WQTM.date()}")
if qprom:
    lead=(WQTM-week_ts(qprom)).days; print(f"  lead vs WQTM                  : {lead} days (~{lead//365}y {(lead%365)//30}m)")
qg=gdf[gdf.q]
if len(qg): print(f"  quantum subgraph : {', '.join(qg.iloc[0].subgraph[:12])}\n  LLM verdict      : {'KEEP' if qg.iloc[0].llm_keep else 'reject/'+str(qg.iloc[0].llm_cat)}")
print("="*68)
print(f"funnel: {R.anchor.nunique():,} novel -> {R[R.caught].anchor.nunique():,} caught -> {len(promoted)} promoted -> {len(groups)} themes -> {int((gdf.llm_keep==True).sum())} LLM-kept")
print("\nLLM-kept themes (shortlist):")
for _,g in gdf[gdf.llm_keep==True].head(15).iterrows():
    print(f"  reach {int(g.reach):3}{' Q' if g.q else '  '}  {', '.join(g.subgraph[:8])}")
R.to_parquet(OUTPUT_DIR/"quantum_entity_anchor_weeks.parquet", index=False)
P.to_parquet(OUTPUT_DIR/"quantum_entity_promotion.parquet", index=False)
print("\nsaved -> quantum_entity_*.parquet")

QUANTUM benchmark (no keywords at detection):
  first CAUGHT (quantum entity) : 2021-03-02/2021-03-08
  PROMOTED                      : 2021-09-21/2021-09-27
  QTUM (broad ETF) 2018-09-04 · WQTM (pure ETF) 2025-10-09
  lead vs WQTM                  : 1479 days (~4y 0m)
  quantum subgraph : a-1, a-2, a-3, a-mark, a-mark precious, a-share, a10, a160m, a320neo, a380, aaa, aaa title
  LLM verdict      : KEEP
funnel: 47,042 novel -> 37,244 caught -> 1312 promoted -> 460 themes -> 2 LLM-kept

LLM-kept themes (shortlist):
  reach 3901 Q  a-1, a-2, a-3, a-mark, a-mark precious, a-share, a10, a160m
  reach 120    abc, act, action, addresses, affairs, afternoon, aggressive, agree

saved -> quantum_entity_*.parquet
